# User-Based Collaborative Filtering
In this notebook, We implement a user-based collaborative filtering
approach using cosine similarity.

Unlike rank-based recommendation, this method provides personalized
recommendations by identifying users with similar rating patterns.

The intuition is that users with similar tastes in the past
are likely to prefer similar books in the future.

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# For evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
base_path = "/content/drive/MyDrive/Goodreads-Book-Recommendation-System/data/"

books = pd.read_csv(base_path + "books.csv")
ratings = pd.read_csv(base_path + "ratings.csv")

print("Books shape:", books.shape)
print("Ratings shape:", ratings.shape)

Books shape: (10000, 23)
Ratings shape: (981756, 3)


# USER–BOOK INTERACTION MATRIX

In [5]:
# Creating user-book interaction matrix

interaction_matrix = ratings.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
).fillna(0)

interaction_matrix.shape

(53424, 10000)

The interaction matrix represents users as rows and books as columns.
Each cell contains the rating given by a user to a book.
Missing ratings are filled with 0.

# COMPUTE USER SIMILARITY

In [ ]:
# Computing cosine similarity between users

#user_similarity = cosine_similarity(interaction_matrix)

#print("User similarity matrix shape:", user_similarity.shape)
#That is:

#≈ 2.85 BILLION similarity values.

#That does NOT fit in Colab RAM.

#It’s a scalability issue.
#Each float ≈ 8 bytes

#That is ~22GB RAM.

#Colab free gives ~12GB.

#So it crashed.

# We Used Subset of Users (For Demonstration)

In [6]:
# To avoid memory issues, we use a subset of users

sample_users = ratings['user_id'].unique()[:3000]

ratings_subset = ratings[ratings['user_id'].isin(sample_users)]

interaction_matrix = ratings_subset.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
).fillna(0)

interaction_matrix.shape

(3000, 9764)

In [7]:
# Computing cosine similarity on subset

from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(interaction_matrix)

print("User similarity matrix shape:", user_similarity.shape)

User similarity matrix shape: (3000, 3000)


The similarity matrix now contains similarity scores between
the selected subset of 3000 users.

Using a subset allows us to demonstrate the working of
user-based collaborative filtering without exhausting system memory.

This highlights a limitation of memory-based methods:
they do not scale efficiently for very large datasets.

#Similar Users Function (Adjusted for Subset)

In [8]:
def get_similar_users(user_id, n=5):
    """
    Returns top N similar users for a given user_id.
    """

    if user_id not in interaction_matrix.index:
        return "User not in selected subset."

    user_index = interaction_matrix.index.get_loc(user_id)

    sim_scores = list(enumerate(user_similarity[user_index]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    return sim_scores[1:n+1]

In [9]:
get_similar_users(sample_users[0])

[(1402, np.float64(0.3245355807582532)),
 (1577, np.float64(0.24436453696942775)),
 (790, np.float64(0.24287355316686385)),
 (119, np.float64(0.233590710227514)),
 (2793, np.float64(0.22261103845226893))]

# Recommendation Function (For the Subset taken earlier)

In [10]:
def recommend_books_user_based(user_id, n=5):
    """
    Recommend books to a user based on similar users' preferences.
    """

    if user_id not in interaction_matrix.index:
        return "User not in selected subset."

    similar_users = get_similar_users(user_id)

    user_books = interaction_matrix.loc[user_id]
    unread_books = user_books[user_books == 0].index

    recommendations = []

    for sim_user, score in similar_users:
        sim_user_id = interaction_matrix.index[sim_user]
        sim_user_books = interaction_matrix.loc[sim_user_id]

        top_books = sim_user_books[sim_user_books > 0].sort_values(ascending=False)

        for book_id in top_books.index:
            if book_id in unread_books:
                recommendations.append(book_id)

            if len(recommendations) >= n:
                break

    recommended_titles = books[
        books['book_id'].isin(recommendations)
    ]['title']

    return recommended_titles

In [11]:
recommend_books_user_based(sample_users[0])

,title
20,Harry Potter and the Order of the Phoenix (Har...
373,A Short History of Nearly Everything
1459,In a Sunburned Country


## Conclusion

In this notebook, We implemented user-based collaborative filtering.

Key observations:
- Personalized recommendations are possible.
- Full similarity computation is memory-intensive.
- Scalability is a limitation of memory-based approaches.

This motivates the use of model-based collaborative filtering methods.